In [2]:
import pandas as pd
import requests
import zipfile
import io
from datetime import datetime, timedelta

In [3]:
import requests
import pandas as pd
import zipfile
import io
from datetime import datetime, timedelta

# ---------------------------------------------
# CONFIGURATION
# ---------------------------------------------

START_DATE = datetime(2018, 6, 1)
END_DATE   = datetime(2018, 12, 31)

headers = {
    "User-Agent": "Mozilla/5.0"
}

all_data = []

# ---------------------------------------------
# LOOP THROUGH EVERY TRADING DAY
# ---------------------------------------------

current = START_DATE

while current <= END_DATE:

    # Skip weekends
    if current.weekday() >= 5:
        current += timedelta(days=1)
        continue

    month = current.strftime("%b").upper()
    filename = f"fo{current.strftime('%d%b%Y').upper()}bhav.csv.zip"

    url = (
        f"https://nsearchives.nseindia.com/"
        f"content/historical/DERIVATIVES/"
        f"{current.year}/{month}/{filename}"
    )

    print(current.strftime("%d-%b-%Y"))

    try:

        r = requests.get(url, headers=headers, timeout=15)

        if r.status_code != 200:
            current += timedelta(days=1)
            continue

        z = zipfile.ZipFile(io.BytesIO(r.content))

        csv_name = z.namelist()[0]

        df = pd.read_csv(z.open(csv_name))

        # Keep only NIFTY Index Futures
        df = df[
            (df["INSTRUMENT"] == "FUTIDX") &
            (df["SYMBOL"] == "NIFTY")
        ].copy()

        if len(df) == 0:
            current += timedelta(days=1)
            continue

        # Convert expiry
        df["EXPIRY_DT"] = pd.to_datetime(df["EXPIRY_DT"])

        # Select nearest expiry (front month)
        nearest = df["EXPIRY_DT"].min()

        df = df[df["EXPIRY_DT"] == nearest]

        df["TRADE_DATE"] = current

        all_data.append(df)

    except Exception:
        pass

    current += timedelta(days=1)

# ---------------------------------------------
# COMBINE EVERYTHING
# ---------------------------------------------

final = pd.concat(all_data, ignore_index=True)

# Keep useful columns

final = final[
    [
        "TRADE_DATE",
        "EXPIRY_DT",
        "SETTLE_PR",
        "OPEN",
        "HIGH",
        "LOW",
        "CLOSE",
        "CONTRACTS",
        "OPEN_INT",
        "VAL_INLAKH"
    ]
]

final.columns = [
    "Date",
    "Expiry",
    "SettlementPrice",
    "Open",
    "High",
    "Low",
    "Close",
    "Contracts",
    "OpenInterest",
    "TurnoverLakhs"
]

final.to_csv("nifty_futures.csv", index=False)

print("\nSaved as nifty_futures.csv")
print(final.head())

01-Jun-2018
04-Jun-2018
05-Jun-2018
06-Jun-2018
07-Jun-2018
08-Jun-2018
11-Jun-2018
12-Jun-2018
13-Jun-2018
14-Jun-2018
15-Jun-2018
18-Jun-2018
19-Jun-2018
20-Jun-2018
21-Jun-2018
22-Jun-2018
25-Jun-2018
26-Jun-2018
27-Jun-2018
28-Jun-2018
29-Jun-2018
02-Jul-2018
03-Jul-2018
04-Jul-2018
05-Jul-2018
06-Jul-2018
09-Jul-2018
10-Jul-2018
11-Jul-2018
12-Jul-2018
13-Jul-2018
16-Jul-2018
17-Jul-2018
18-Jul-2018
19-Jul-2018
20-Jul-2018
23-Jul-2018
24-Jul-2018
25-Jul-2018
26-Jul-2018
27-Jul-2018
30-Jul-2018
31-Jul-2018
01-Aug-2018
02-Aug-2018
03-Aug-2018
06-Aug-2018
07-Aug-2018
08-Aug-2018
09-Aug-2018
10-Aug-2018
13-Aug-2018
14-Aug-2018
15-Aug-2018
16-Aug-2018
17-Aug-2018
20-Aug-2018
21-Aug-2018
22-Aug-2018
23-Aug-2018
24-Aug-2018
27-Aug-2018
28-Aug-2018
29-Aug-2018
30-Aug-2018
31-Aug-2018
03-Sep-2018
04-Sep-2018
05-Sep-2018
06-Sep-2018
07-Sep-2018
10-Sep-2018
11-Sep-2018
12-Sep-2018
13-Sep-2018
14-Sep-2018
17-Sep-2018
18-Sep-2018
19-Sep-2018
20-Sep-2018
21-Sep-2018
24-Sep-2018
25-Sep-2018
26-S

In [4]:
import os

print(os.getcwd())

c:\Users\sanka\AppData\Local\Programs\Microsoft VS Code


In [12]:
# ============================================================
# NSE NIFTY OPTIONS DOWNLOADER (ONE MONTH)
# MSc Project - IL&FS Crisis
# Downloads ONLY NIFTY Index Options (OPTIDX)
# ============================================================

import os
import io
import time
import random
import zipfile
import requests
import pandas as pd

from datetime import datetime, timedelta

# ============================================================
# CONFIGURATION
# ============================================================

YEAR = 2018
MONTH = 12        # Change to 7,8,9,10,11,12

SAVE_FOLDER = "data/raw"

os.makedirs(SAVE_FOLDER, exist_ok=True)

BASE_URL = "https://nsearchives.nseindia.com/content/historical/DERIVATIVES"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/126.0.0.0 Safari/537.36"
    ),
    "Accept": "*/*",
    "Accept-Language": "en-US,en;q=0.9",
    "Connection": "keep-alive"
}

# ============================================================
# DETERMINE MONTH START & END
# ============================================================

start_date = datetime(YEAR, MONTH, 1)

if MONTH == 12:
    end_date = datetime(YEAR + 1, 1, 1) - timedelta(days=1)
else:
    end_date = datetime(YEAR, MONTH + 1, 1) - timedelta(days=1)

# ============================================================
# CREATE SESSION
# ============================================================

session = requests.Session()
session.headers.update(HEADERS)

all_data = []
failed_dates = []

current = start_date

print("=" * 70)
print(f"Downloading NIFTY OPTIONS : {start_date.strftime('%B %Y')}")
print("=" * 70)

while current <= end_date:

    # Skip Saturday & Sunday
    if current.weekday() >= 5:
        current += timedelta(days=1)
        continue

    month_abbr = current.strftime("%b").upper()

    filename = f"fo{current.strftime('%d%b%Y').upper()}bhav.csv.zip"

    url = (
        f"{BASE_URL}/"
        f"{YEAR}/"
        f"{month_abbr}/"
        f"{filename}"
    )

    print(current.strftime("%d-%b-%Y"), end=" --> ")

    try:

        response = session.get(url, timeout=30)

        if response.status_code != 200:
            print("Holiday / File Not Found")
            failed_dates.append(current.strftime("%Y-%m-%d"))
            current += timedelta(days=1)
            continue

        zip_file = zipfile.ZipFile(io.BytesIO(response.content))

        csv_name = zip_file.namelist()[0]

        df = pd.read_csv(zip_file.open(csv_name))

        # ----------------------------------------------------
        # KEEP ONLY NIFTY INDEX OPTIONS
        # ----------------------------------------------------

        df = df[
            (df["INSTRUMENT"] == "OPTIDX") &
            (df["SYMBOL"] == "NIFTY")
        ].copy()

        if df.empty:
            print("No NIFTY Options")
            current += timedelta(days=1)
            continue

        df["TRADE_DATE"] = current

        all_data.append(df)

        print(f"{len(df):,} rows")

        # Random delay (prevents rate limiting)
        time.sleep(random.uniform(2, 4))

    except Exception as e:

        print("FAILED")
        failed_dates.append(current.strftime("%Y-%m-%d"))

    current += timedelta(days=1)

# ============================================================
# SAVE MONTHLY DATA
# ============================================================

if all_data:

    final = pd.concat(all_data, ignore_index=True)

    month_name = start_date.strftime("%B").lower()

    output_file = os.path.join(
        SAVE_FOLDER,
        f"nifty_options_{month_name}_2018.csv"
    )

    final.to_csv(output_file, index=False)

    print("\n" + "=" * 70)
    print("DOWNLOAD COMPLETED")
    print("=" * 70)
    print("Saved File :", output_file)
    print("Rows       :", f"{len(final):,}")

else:

    print("\nNo data downloaded.")

# ============================================================
# SAVE FAILED DAYS
# ============================================================

if failed_dates:

    failed_file = os.path.join(
        SAVE_FOLDER,
        f"failed_options_{MONTH}.csv"
    )

    pd.DataFrame({
        "FAILED_DATE": failed_dates
    }).to_csv(failed_file, index=False)

    print("\nFailed Dates saved to:", failed_file)

print("\nFinished.")

03-Dec-2018 --> 2,340 rows
04-Dec-2018 --> 2,340 rows
05-Dec-2018 --> 2,340 rows
06-Dec-2018 --> 2,342 rows
07-Dec-2018 --> 2,350 rows
10-Dec-2018 --> 2,350 rows
11-Dec-2018 --> 2,354 rows
12-Dec-2018 --> 2,354 rows
13-Dec-2018 --> 2,354 rows
14-Dec-2018 --> 2,354 rows
17-Dec-2018 --> 2,354 rows
18-Dec-2018 --> 2,354 rows
19-Dec-2018 --> 2,354 rows
20-Dec-2018 --> 2,358 rows
21-Dec-2018 --> 2,358 rows
24-Dec-2018 --> 2,358 rows
25-Dec-2018 --> Holiday / File Not Found
26-Dec-2018 --> 2,358 rows
27-Dec-2018 --> 2,358 rows
28-Dec-2018 --> 2,224 rows
31-Dec-2018 --> 2,228 rows

DOWNLOAD COMPLETED
Saved File : data/raw\nifty_options_december_2018.csv
Rows       : 46,782

Failed Dates saved to: data/raw\failed_options_12.csv

Finished.


In [13]:
import yfinance as yf
import pandas as pd
import os

# ---------------------------------------------------
# Configuration
# ---------------------------------------------------

START_DATE = "2018-06-01"
END_DATE   = "2019-01-01"   # End date is exclusive

SAVE_FOLDER = "data/raw"
os.makedirs(SAVE_FOLDER, exist_ok=True)

# ---------------------------------------------------
# Download India VIX
# ---------------------------------------------------

vix = yf.download(
    "^INDIAVIX",
    start=START_DATE,
    end=END_DATE,
    auto_adjust=False,
    progress=True
)

# Reset index so Date becomes a column
vix.reset_index(inplace=True)

# Save CSV
output_file = os.path.join(SAVE_FOLDER, "india_vix.csv")
vix.to_csv(output_file, index=False)

print("=" * 60)
print("India VIX downloaded successfully!")
print("=" * 60)
print(f"Rows : {len(vix)}")
print(f"Saved: {output_file}")

print("\nPreview:")
print(vix.head())

[*********************100%***********************]  1 of 1 completed

India VIX downloaded successfully!
Rows : 143
Saved: data/raw\india_vix.csv

Preview:
Price        Date Adj Close     Close      High       Low      Open    Volume
Ticker            ^INDIAVIX ^INDIAVIX ^INDIAVIX ^INDIAVIX ^INDIAVIX ^INDIAVIX
0      2018-06-01     13.57     13.57     14.59     12.16     13.22         0
1      2018-06-04     13.88     13.88     15.47     12.18     13.57         0
2      2018-06-05     13.32     13.32     14.49     13.13     13.88         0
3      2018-06-06     12.55     12.55     13.51     12.15     13.32         0
4      2018-06-07     12.78     12.78     13.65     11.61     12.55         0


In [6]:
import os
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd
from nselib import derivatives

# ============================================================
# CONFIGURATION
# ============================================================

START_DATE = "01-06-2018"
END_DATE   = "31-12-2018"

OUTPUT_DIR = Path("Data/Raw")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# DOWNLOAD BANKNIFTY FUTURES
# ============================================================

print("Downloading BANKNIFTY Futures...")

banknifty_futures = derivatives.future_price_volume_data(
    symbol="BANKNIFTY",
    instrument="FUTIDX",
    from_date=START_DATE,
    to_date=END_DATE
)

# Save raw file
output_file = OUTPUT_DIR / "banknifty_futures.csv"
banknifty_futures.to_csv(output_file, index=False)

print(f"Downloaded {len(banknifty_futures):,} rows")
print(f"Saved to: {output_file}")

# Preview
banknifty_futures.head()

Downloaded 436 rows
Saved to: Data\Raw\banknifty_futures.csv


,INSTRUMENT,SYMBOL,EXPIRY_DT,STRIKE_PRICE,OPTION_TYPE,MARKET_TYPE,OPENING_PRICE,TRADE_HIGH_PRICE,TRADE_LOW_PRICE,CLOSING_PRICE,...,SETTLE_PRICE,TOT_TRADED_QTY,TOT_TRADED_VAL,OPEN_INT,CHANGE_IN_OI,MARKET_LOT,TIMESTAMP,TIMESTAMP_ORDER,UNDERLYING_VALUE,CALCULATED_PREMIUM_VAL
0,FUTIDX,BANKNIFTY,29-Nov-2018,0,FF,None,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,30-Aug-2018,2018-08-29T18:30:00.000Z,NaN,NaN
1,FUTIDX,BANKNIFTY,27-Sep-2018,0,XX,N,28343.20,28343.20,28124.10,28221.85,...,28221.85,1856920.0,523466.12,1646720.0,596120.0,40.0,30-Aug-2018,2018-08-29T18:30:00.000Z,NaN,523466.12
2,FUTIDX,BANKNIFTY,30-Aug-2018,0,XX,N,28233.30,28233.30,27966.45,28094.15,...,28103.25,3259480.0,914658.71,862440.0,-458360.0,40.0,30-Aug-2018,2018-08-29T18:30:00.000Z,NaN,914658.71
3,FUTIDX,BANKNIFTY,25-Oct-2018,0,XX,N,28320.65,28323.65,28191.65,28271.35,...,28271.35,16840.0,4756.51,21560.0,3200.0,40.0,30-Aug-2018,2018-08-29T18:30:00.000Z,NaN,4756.51
4,FUTIDX,BANKNIFTY,25-Oct-2018,0,XX,N,28409.65,28530.00,28365.00,28412.85,...,28412.85,7840.0,2230.40,18360.0,2960.0,40.0,29-Aug-2018,2018-08-28T18:30:00.000Z,28224.1,2230.40
